In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score
from sklearn.feature_selection import SelectKBest, f_regression
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

df = pd.read_csv("github_repo_features.csv")

df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
df["updated_at"] = pd.to_datetime(df["updated_at"]).dt.tz_localize(None)
df["pushed_at"] = pd.to_datetime(df["pushed_at"]).dt.tz_localize(None)
ref_date = datetime(2025, 5, 1)

df["project_age"] = (ref_date - df["created_at"]).dt.days
df["days_since_update"] = (ref_date - df["updated_at"]).dt.days
df["days_since_push"] = (ref_date - df["pushed_at"]).dt.days

df["forks_per_day"] = df["forks"] / (df["project_age"] + 1)
df["issues_per_day"] = df["open_issues"] / (df["project_age"] + 1)
df["update_rate"] = 1 / (1 + df["days_since_update"])

#New features based on correlation matrix
df["stars_per_fork"] = df["stars"] / (df["forks"] + 1)
df["stars_per_commit"] = df["stars"] / (df["commits_count"] + 1)
df["activity_level"] = df["commits_count"] + df["open_issues"] + df["subscribers_count"]
df["recent_activity"] = df["update_rate"] * df["commits_count"]
df["forks_times_commits"] = df["forks"] * df["commits_count"]

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

for col in ["has_wiki", "has_projects", "has_downloads", "is_fork", "archived"]:
    df[col] = df[col].astype(int)

features = [
    'open_issues', 'size', 'has_wiki', 'has_projects', 'has_downloads',
    'is_fork', 'archived', 'language', 'license', 'subscribers_count',
    'contributors_count', 'commits_count', 'readme_size', 'project_age',
    'days_since_update', 'days_since_push', 'forks_per_day', 'update_rate',
    'stars_per_fork', 'stars_per_commit', 'activity_level', 'recent_activity',
    'forks_times_commits'
]

X = df[features]
y = df["stars"]

#preprocessing
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

#define model
model = Sequential()
model.add(tf.keras.Input(shape=(X_train.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(1))

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])

#train the model
model.fit(X_train, y_train, epochs=150, batch_size=10, validation_split=0.2)

model.save('neural_network_new_model.h5')

predictions = model.predict(X_test)
print('R2 score:', r2_score(y_test, predictions))


Epoch 1/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 3506070784.0000 - mae: 48097.6641 - val_loss: 4259936512.0000 - val_mae: 52325.9961
Epoch 2/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3825924864.0000 - mae: 49487.6445 - val_loss: 4257177856.0000 - val_mae: 52302.2070
Epoch 3/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3171754496.0000 - mae: 47809.4336 - val_loss: 4248092928.0000 - val_mae: 52225.2188
Epoch 4/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3600492032.0000 - mae: 48252.1641 - val_loss: 4227693824.0000 - val_mae: 52054.6680
Epoch 5/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3500963072.0000 - mae: 47848.5977 - val_loss: 4189942272.0000 - val_mae: 51745.3047
Epoch 6/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3392831744.0000 - mae: 47025.9414 - val_loss: 4131061760.0000 - val_mae: 51263.5039
Epoch 7/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3018094336.0000 - mae: 46076.9219 - val_loss: 4049816064.0000 - val_mae: 50587.839

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
R2 score: 0.9328818917274475
